# Treinamento de modelo de turnover

Este notebook lê exclusivamente `data/cleaned/turnover_features.csv`. A base contém somente eventos limitados à data de referência individual; identificador, alvo e data de referência nunca entram como features.

**Uso responsável:** o modelo é um experimento analítico de propensão, não pode automatizar decisões de RH. As métricas são avaliadas em período futuro, e o artefato inclui as features e a versão das bibliotecas para reprodutibilidade.

In [1]:
import joblib
import numpy as np
import pandas as pd
import sklearn
from pathlib import Path
from IPython.display import display
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, average_precision_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

RANDOM_STATE = 42
caminho_cleaned = Path('data/cleaned/turnover_features.csv')
base = pd.read_csv(caminho_cleaned, parse_dates=['data_referencia'])
base['data_referencia'] = pd.to_datetime(base['data_referencia'])
print(base.shape)
base.head()


(24000, 32)


,nIdPessoa,data_referencia,pediu_para_sair,acidentes_eventos,acidentes_com_afastamento,acidentes_dias_perdidos,abs_eventos,abs_qtd_total,horas_previstas_total,he_eventos,...,teve_hora_extra,dias_desde_ultima_hora_extra,teve_hora_irregular,dias_desde_ultima_hora_irregular,teve_mov_sal,dias_desde_ultimo_mov_sal,taxa_absenteismo,taxa_hora_irregular,he_valor_medio,mov_sal_valor_medio
0,181076,2018-06-10,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,-1,0,-1,0,-1,0.0,0.0,0.000000,0.000000
1,201073,2019-02-17,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,-1,0,-1,0,-1,0.0,0.0,0.000000,0.000000
2,280267,2023-05-03,1,0.0,0.0,0.0,0.0,0.0,0.0,34.0,...,1,153,0,-1,1,214,0.0,0.0,56.017941,166.493710
3,14686,2017-05-03,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0,-1,0,-1,1,63,0.0,0.0,0.000000,59.651065
4,71442,2021-04-09,0,0.0,0.0,0.0,0.0,0.0,0.0,49.0,...,1,8,1,492,1,190,0.0,0.0,27.229184,52.658537


## 1. Validações de contrato e seleção de features

A validação garante que o arquivo de entrada não carregue atributos cadastrais, dados futuros ou identificadores para dentro do modelo.

In [2]:
colunas_proibidas = {
    'nIdPessoa', 'pediu_para_sair', 'data_referencia', 'dAnoMes', 'cSituacao',
    'cSexo', 'cCor', 'cEstadoCivil', 'nIdade', 'cGeracao', 'cCidade', 'cEstado',
    'nSalarioTotal', 'nRemuneracaoTotal', 'nTempoDeCasaAnos',
}
assert {'nIdPessoa', 'pediu_para_sair', 'data_referencia'}.issubset(base.columns)
assert not (colunas_proibidas - {'nIdPessoa', 'pediu_para_sair', 'data_referencia'}).intersection(base.columns)
assert base['nIdPessoa'].is_unique
assert base['pediu_para_sair'].isin([0, 1]).all()

features = [coluna for coluna in base.columns if coluna not in {'nIdPessoa', 'pediu_para_sair', 'data_referencia'}]
assert features
print(f'Features usadas ({len(features)}):')
print(features)


Features usadas (29):
['acidentes_eventos', 'acidentes_com_afastamento', 'acidentes_dias_perdidos', 'abs_eventos', 'abs_qtd_total', 'horas_previstas_total', 'he_eventos', 'he_referencia_total', 'he_valor_total', 'hi_eventos', 'hi_minutos_irregulares', 'hi_minutos_extras', 'mov_sal_eventos', 'mov_sal_valor_total', 'mov_sal_perc_medio', 'teve_acidente', 'dias_desde_ultimo_acidente', 'teve_absenteismo', 'dias_desde_ultimo_absenteismo', 'teve_hora_extra', 'dias_desde_ultima_hora_extra', 'teve_hora_irregular', 'dias_desde_ultima_hora_irregular', 'teve_mov_sal', 'dias_desde_ultimo_mov_sal', 'taxa_absenteismo', 'taxa_hora_irregular', 'he_valor_medio', 'mov_sal_valor_medio']


## 2. Split temporal, baseline e regressão logística

Amostras até 2022 são usadas para treino e 2023 é reservado integralmente para teste. A coorte de 2024 não entra: ela contém somente pessoas ainda ativas e não tem rótulo de saída confirmado. A regressão logística L2 regularizada não usa árvores; imputação e padronização são aprendidas exclusivamente no treino. O limiar é escolhido com predições out-of-fold do treino, sem consultar o teste.

In [3]:
base_modelavel = base[base['data_referencia'].dt.year <= 2023].copy()
treino = base_modelavel[base_modelavel['data_referencia'].dt.year <= 2022].copy()
teste = base_modelavel[base_modelavel['data_referencia'].dt.year == 2023].copy()
X_train, y_train = treino[features], treino['pediu_para_sair']
X_test, y_test = teste[features], teste['pediu_para_sair']
assert y_train.nunique() == 2 and y_test.nunique() == 2
print(f'Treino: {len(treino):,} | saída voluntária: {y_train.mean():.1%}')
print(f'Teste: {len(teste):,} | saída voluntária: {y_test.mean():.1%}')

pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('modelo', LogisticRegression(
        penalty='l2', C=1.0, class_weight='balanced', solver='liblinear',
        max_iter=2_000, random_state=RANDOM_STATE,
    )),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_pr_auc = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='average_precision', n_jobs=-1)
proba_oof = cross_val_predict(pipeline, X_train, y_train, cv=cv, method='predict_proba', n_jobs=-1)[:, 1]
limiares = np.linspace(0.05, 0.95, 181)
LIMIAR = max(limiares, key=lambda limiar: f1_score(y_train, proba_oof >= limiar))
baseline = DummyClassifier(strategy='prior').fit(X_train, y_train)
pipeline.fit(X_train, y_train)
print(f'PR-AUC CV no treino: {cv_pr_auc.mean():.3f} ± {cv_pr_auc.std():.3f}')
print(f'Limiar escolhido por CV no treino: {LIMIAR:.2f}')


Treino: 17,560 | saída voluntária: 58.3%
Teste: 3,043 | saída voluntária: 68.6%
PR-AUC CV no treino: 0.663 ± 0.004
Limiar escolhido por CV no treino: 0.21


## 3. Avaliação no teste temporal

PR-AUC é a métrica principal para qualidade de ranking da classe positiva. Precision, recall e F1 usam o limiar selecionado por validação cruzada no treino; o teste não participa dessa escolha. O limiar deve ser revisado com RH antes de qualquer uso operacional.

In [4]:
proba = pipeline.predict_proba(X_test)[:, 1]
predicao = (proba >= LIMIAR).astype(int)
proba_baseline = baseline.predict_proba(X_test)[:, 1]
metricas = pd.DataFrame({
    'modelo': ['Baseline (prior)', 'Regressão Logística'],
    'ROC-AUC': [roc_auc_score(y_test, proba_baseline), roc_auc_score(y_test, proba)],
    'PR-AUC': [average_precision_score(y_test, proba_baseline), average_precision_score(y_test, proba)],
    'Precision @ limiar CV': [precision_score(y_test, (proba_baseline >= LIMIAR).astype(int), zero_division=0), precision_score(y_test, predicao, zero_division=0)],
    'Recall @ limiar CV': [recall_score(y_test, (proba_baseline >= LIMIAR).astype(int), zero_division=0), recall_score(y_test, predicao, zero_division=0)],
    'F1 @ limiar CV': [f1_score(y_test, (proba_baseline >= LIMIAR).astype(int), zero_division=0), f1_score(y_test, predicao, zero_division=0)],
}).set_index('modelo').round(3)
display(metricas)
print(classification_report(y_test, predicao, digits=3, zero_division=0))
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_test, predicao), display_labels=['Não voluntário', 'Voluntário']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Matriz de confusão — teste temporal 2023')
plt.tight_layout()
plt.show()


,ROC-AUC,PR-AUC,Precision @ limiar CV,Recall @ limiar CV,F1 @ limiar CV
modelo,,,,,
Baseline (prior),0.500,0.686,0.686,1.000,0.814
Regressão Logística,0.639,0.762,0.699,0.989,0.819


              precision    recall  f1-score   support

           0      0.733     0.066     0.121       954
           1      0.699     0.989     0.819      2089

    accuracy                          0.700      3043
   macro avg      0.716     0.528     0.470      3043
weighted avg      0.709     0.700     0.600      3043



## 4. Contribuição das features

A importância por permutação mede a queda de PR-AUC ao embaralhar cada feature no conjunto de teste. Ela mostra associação preditiva, não causalidade.

In [5]:
permutacao = permutation_importance(
    pipeline, X_test, y_test, scoring='average_precision', n_repeats=10,
    random_state=RANDOM_STATE, n_jobs=-1,
)
importancias = pd.DataFrame({
    'feature': features,
    'queda_media_pr_auc': permutacao.importances_mean,
    'desvio_padrao': permutacao.importances_std,
}).sort_values('queda_media_pr_auc', ascending=False).reset_index(drop=True)
display(importancias)


,feature,queda_media_pr_auc,desvio_padrao
0,he_valor_total,0.082485,0.003742
1,mov_sal_eventos,0.036013,0.005414
2,he_referencia_total,0.028007,0.006205
3,teve_hora_extra,0.019629,0.004362
4,he_eventos,0.007690,0.007146
5,horas_previstas_total,0.005634,0.000266
6,mov_sal_valor_medio,0.005576,0.004318
7,teve_hora_irregular,0.002620,0.002608
8,hi_eventos,0.002418,0.001795
9,teve_absenteismo,0.002124,0.000264


## 5. Persistência reprodutível

O artefato contém pipeline, lista de features, períodos, métricas e versão do scikit-learn. O identificador e o alvo não são persistidos como entradas do modelo.

In [6]:
artefato = {
    'pipeline': pipeline,
    'features': features,
    'target': 'pediu_para_sair',
    'threshold': LIMIAR,
    'train_period': 'até 2022',
    'test_period': '2023',
    'excluded_censored_period': '2024',
    'test_metrics': metricas.loc['Regressão Logística'].to_dict(),
    'cv_pr_auc_mean': float(cv_pr_auc.mean()),
    'cv_pr_auc_std': float(cv_pr_auc.std()),
    'sklearn_version': sklearn.__version__,
    'model_type': 'LogisticRegression',
    'model_version': '1.0.0-event-features-only',
}
caminho_modelo = Path('artifacts/turnover_logistic_event_features_v1.pkl')
caminho_modelo.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(artefato, caminho_modelo)
recarregado = joblib.load(caminho_modelo)
assert np.allclose(recarregado['pipeline'].predict_proba(X_test.head(20)), pipeline.predict_proba(X_test.head(20)))
print(f'Modelo salvo e validado: {caminho_modelo.resolve()}')


Modelo salvo e validado: C:\Users\miguelaraujo-ieg\OneDrive - Instituto Germinare\3°ANO\Ciencia de Dados 2\Turnover\turnover\artifacts\turnover_logistic_event_features_v1.pkl
